In [ ]:
# 导入库

import os
import sympy
import sympy.printing

os.environ["TORCHDYNAMO_DISABLE"] = "1" 

import torch
import torch.nn as nn
import torchaudio
import torchaudio.transforms as T
import torch.nn.functional as F
import pandas as pd
import xml.etree.ElementTree as ET
from torch.utils.data import Dataset, DataLoader

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
def pad_collate(batch):
    """
    自定义的 batch 整理函数：将同一个 batch 内的不同长度的频谱图补齐到最大长度。
    """
    is_train = len(batch[0]) == 3
    
    max_len = max([item[0].shape[-1] for item in batch])
    
    padded_specs = []
    labels = []
    file_names = []
    
    for item in batch:
        spec = item[0]
        pad_amount = max_len - spec.shape[-1]
        
        padded_spec = F.pad(spec, (0, pad_amount), mode='constant', value=-80.0)
        padded_specs.append(padded_spec)
        
        if is_train:
            labels.append(item[1])
            file_names.append(item[2])
        else:
            file_names.append(item[1])
            
    padded_specs = torch.stack(padded_specs)
    
    if is_train:
        labels = torch.stack(labels)
        return padded_specs, labels, file_names
    else:
        return padded_specs, file_names

In [ ]:
class DrumDataset(Dataset):
    def __init__(self, audio_dir, xml_dir=None, is_train=True):
        self.audio_dir = audio_dir
        self.xml_dir = xml_dir
        self.is_train = is_train
        self.file_list = [f for f in os.listdir(audio_dir) if f.endswith('.wav') and not f.startswith('.')]

        raw_files = [f for f in os.listdir(audio_dir) if f.endswith('.wav') and not f.startswith('.')]
        self.file_list = sorted(raw_files)
        
        self.mel_spectrogram = T.MelSpectrogram(
            sample_rate=44100,
            n_fft=2048,
            hop_length=512,
            n_mels=64
        )
        self.amplitude_to_db = T.AmplitudeToDB()

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        file_name = self.file_list[idx]
        audio_path = os.path.join(self.audio_dir, file_name)
        
        waveform, sample_rate = torchaudio.load(audio_path)
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
            
        mel_spec = self.mel_spectrogram(waveform)
        mel_spec = self.amplitude_to_db(mel_spec) 
        
        if self.is_train and self.xml_dir:
            xml_path = os.path.join(self.xml_dir, file_name.replace('.wav', '.xml'))
            kd_count, sd_count = self._parse_xml(xml_path)
            labels = torch.tensor([kd_count, sd_count], dtype=torch.float32)
            return mel_spec, labels, file_name
        
        return mel_spec, file_name

    def _parse_xml(self, xml_path):
        """解析官方 XML 标签，统计 KD 和 SD 的出现次数"""
        try:
            tree = ET.parse(xml_path)
            root = tree.getroot()
            
            kd_count = 0
            sd_count = 0
            
            for event in root.findall(".//event"):
                instrument = event.find("instrument")
                if instrument is not None:
                    if instrument.text == "KD":
                        kd_count += 1
                    elif instrument.text == "SD":
                        sd_count += 1
                        
            return kd_count, sd_count
            
        except Exception as e:
            print(f"解析 {xml_path} 失败: {e}")
            return 0, 0


In [ ]:
class DrumCounterNet(nn.Module):
    def __init__(self):
        super(DrumCounterNet, self).__init__()
        
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        
        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 2)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.regressor(x)
        return x

In [ ]:
def train_model(model, train_loader, num_epochs=50, lr=0.001):
    criterion = nn.L1Loss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    model.train()
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        for specs, labels, _ in train_loader:
            specs, labels = specs.to(device), labels.to(device)
            
            outputs = model(specs)
            loss = criterion(outputs, labels)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            
        if (epoch + 1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}], Loss (MAE): {epoch_loss/len(train_loader):.4f}")
    
    return model

def generate_submission(model, test_loader, output_file="submissionA.csv"):
    model.eval()
    results = []
    
    with torch.no_grad():
        for specs, _ in test_loader:
            specs = specs.to(device)
            preds = model(specs)

            preds = torch.clamp(torch.round(preds), min=0).int()

            for i in range(preds.shape[0]):
                results.append([preds[i][0].item(), preds[i][1].item()])
                
    df = pd.DataFrame(results)
    df.to_csv(output_file, index=False, header=False)
    print(f"提交文件 {output_file} 已生成")

In [ ]:
model = DrumCounterNet().to(device)
train_dataset = DrumDataset("/kaggle/working/noai-train-drum-count/dataset/training_set/audio_mix", "/kaggle/working/noai-train-drum-count/dataset/training_set/annotation_xml")
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=pad_collate)
trained_model = train_model(model, train_loader, num_epochs=50)

In [ ]:
val_dataset = DrumDataset("/kaggle/working/noai-train-drum-count/dataset/validation_set/audio_mix", is_train=False)
test_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, collate_fn=pad_collate)
generate_submission(trained_model, test_loader, "submissionA.csv")

test_dataset = DrumDataset("/kaggle/working/noai-train-drum-count/dataset/testing_set/audio_mix", is_train=False)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, collate_fn=pad_collate)
generate_submission(trained_model, test_loader, "submissionB.csv")

import zipfile
with zipfile.ZipFile("submission.zip", "w") as zipf:
    zipf.write("submissionA.csv")
    zipf.write("submissionB.csv")
os.remove("submissionA.csv")
os.remove("submissionB.csv")